# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

## Step 16 / 18b scratch — baseline OCR numbers (pretrained, not fine-tuned)

Full-book run on Kaggle GPU T4×2 (`facebook/nougat-base`, **not** fine-tuned —
fine-tuning is Sprint 4 / Step 28). `KAGGLE/step18b_ocr_repair/kaggle_step18b_ocr_repair.ipynb`
produced this output (the Step 18b repaired-reader re-run, 2026-08-10). Preserved at
`data/old baseline ocr/` (2026-08-13, ahead of Step 30): Step 30's fine-tuned re-OCR writes
directly into `data/ocr/`, replacing it, so the pretrained BEFORE snapshot this section
reports would silently drift into the fine-tuned AFTER result if it stayed there. The cells
below read the preserved copy directly, so every number is still reproducible from this
repo's own state, not copied by hand -- just no longer from `data/ocr/` itself.

Step 16's baseline missed most of the book (15.1% word coverage); Step 18b fixed three
inference-path bugs and re-ran the full book. The gate below is plan.md Step 18b's own
pre-committed decision rule (no-output rate ≤25% → keep the train set; >25% → reopen the
annotation budget, evidence-based) — it did **not** pass, so the train set was expanded
105→122 pages, targeted at the chapters the repaired reader still failed hardest on. See
`plan.md` Step 18b for the full evidence and decision record.

This is the **BEFORE** number Step 29 will compare the fine-tuned reader against.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "../src")  # notebook runs from notebooks/; doc_agent lives in ../src
from doc_agent.eval import metrics

# Frozen Step 18b baseline snapshot -- Step 30's fine-tuned re-OCR writes into
# ../data/ocr directly, so this section reads the preserved pre-fine-tune copy instead
# (moved there 2026-08-13, ahead of Step 30; see .gitignore + plan.md Step 30).
OCR_DIR = Path("../data/old baseline ocr")
LABELS_PATH = Path("../grading_kit/labels.jsonl")
N_CONTENT_PAGES = 1040  # ingest/loader.py: as_p* pages after dropping blanks/front matter

failures = json.loads((OCR_DIR / "failures.json").read_text(encoding="utf-8"))
mmd_files = sorted(OCR_DIR.glob("*.mmd"))
by_reason: dict[str, int] = {}
for row in failures:
    by_reason[row["reason"]] = by_reason.get(row["reason"], 0) + 1

print(f"pages attempted      : {N_CONTENT_PAGES}")
print(f"transcripts produced : {len(mmd_files)}")
print(f"failed / degenerate  : {len(failures)}  ({100 * len(failures) / N_CONTENT_PAGES:.1f}%)")
for reason, count in sorted(by_reason.items(), key=lambda kv: -kv[1]):
    print(f"   {reason:<28} {count:>4}  ({100 * count / len(failures):.1f}% of failures)")

words = sum(len(p.read_text(encoding="utf-8").split()) for p in mmd_files)
print(f"\nwords from OUR OCR   : {words}  (task.yaml floor: 60,000 -- {'MET' if words >= 60000 else 'NOT MET'})")

gold: dict[str, str] = {}
for line in LABELS_PATH.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line:
        row = json.loads(line)
        gold[row["page_id"]] = row["text"]

print(f"\n{'page':<10} {'char-F1':>9} {'exact-form':>12} {'gold-form':>10} {'pred-form':>10}  status")
print("-" * 72)
failed_reason = {row["page_id"]: row["reason"] for row in failures}
scored = []
for pid in ("as_p0243", "as_p0255", "as_p0360"):
    mmd = OCR_DIR / f"{pid}.mmd"
    if not mmd.exists():
        reason = failed_reason.get(pid, "?")
        print(f"{pid:<10} {'--':>9} {'--':>12} {'--':>10} {'--':>10}  FAILED: {reason}")
        continue
    pred = mmd.read_text(encoding="utf-8")
    g = gold[pid]
    f1 = metrics.ocr_f1(pred, g)
    ex = metrics.exact_formula_match(pred, g)
    ngf = len(metrics.extract_formulas(g))
    npf = len(metrics.extract_formulas(pred))
    print(f"{pid:<10} {f1:>9.4f} {ex:>12.4f} {ngf:>10} {npf:>10}  ok")
    scored.append((pid, f1, ex, ngf))
print("-" * 72)

if scored:
    mean_f1 = sum(s[1] for s in scored) / len(scored)
    total_f = sum(s[3] for s in scored)
    weighted_ex = (
        sum(s[2] * s[3] for s in scored) / total_f if total_f else 0.0
    )
    print(f"mean char-F1 (n={len(scored)} gold page(s) with a transcript): {mean_f1:.4f}")
    print(f"exact-formula-match, weighted by formula count: {weighted_ex:.4f}")

print("\nWe expected  the FINE-TUNED reader to land at char-F1 0.88-0.93,")
print("exact-match 0.55-0.75. This is the pretrained BEFORE number Step 29 compares against.")


pages attempted      : 1040
transcripts produced : 744
failed / degenerate  : 296  (28.5%)
   empty-or-near-empty           175  (59.1% of failures)
   nougat-missing-page-marker     73  (24.7% of failures)
   repetition-degeneration        48  (16.2% of failures)

words from OUR OCR   : 161941  (task.yaml floor: 60,000 -- MET)

page         char-F1   exact-form  gold-form  pred-form  status
------------------------------------------------------------------------
as_p0243      0.3360       0.0000          0          0  ok
as_p0255      0.2446       0.0000         14          3  ok
as_p0360      0.5910       0.0000         13         18  ok
------------------------------------------------------------------------
mean char-F1 (n=3 gold page(s) with a transcript): 0.3905
exact-formula-match, weighted by formula count: 0.0000

We expected  the FINE-TUNED reader to land at char-F1 0.88-0.93,
exact-match 0.55-0.75. This is the pretrained BEFORE number Step 29 compares against.


In [2]:
# --- Step 18b: the BEFORE/AFTER table + the PDF-text-layer coverage metric the cell above
# doesn't compute (that's the diagnostic that actually found this bug -- see plan.md
# Step 18b). BEFORE numbers are Step 16's measured baseline, hardcoded because the pre-repair
# OCR output no longer exists anywhere to recompute them from (same pattern as the Kaggle
# notebook's own BEFORE_REPAIR constant). Everything under AFTER is computed from cell
# above's mmd_files/words -- reproducible from the frozen data/old baseline ocr/ snapshot
# (2026-08-13: was ../data/ocr, until Step 30's fine-tuned re-OCR started writing there
# instead), not copied by hand.
import pymupdf

PDF_PATH = Path("../data/raw/handbookofmathem1964abra.pdf")
FRONT_MATTER_OFFSET = 32  # printed N = PDF N + 32 (scripts/get_data.sh)

BEFORE_REPAIR = {
    "no_output_rate": 0.429,
    "median_page_coverage": 0.28,
    "book_wide_coverage": 0.151,
    "as_p0360_precision": 0.813,
    "as_p0360_recall": 0.280,
}

_pdf_doc = pymupdf.open(PDF_PATH)


def _pdf_word_count(printed_page: int) -> int | None:
    idx = printed_page + FRONT_MATTER_OFFSET - 1
    if idx < 0 or idx >= _pdf_doc.page_count:
        return None
    return len(_pdf_doc.load_page(idx).get_text("text").split())


per_page_coverage = []
for f in mmd_files:
    pid = f.stem
    if not pid.startswith("as_p"):
        continue
    pdf_words = _pdf_word_count(int(pid[4:]))
    if not pdf_words:
        continue
    ocr_words = len(f.read_text(encoding="utf-8").split())
    per_page_coverage.append(ocr_words / pdf_words)
per_page_coverage.sort()
median_coverage = per_page_coverage[len(per_page_coverage) // 2] if per_page_coverage else 0.0

book_wide_pdf_words = sum(
    len(_pdf_doc.load_page(i).get_text("text").split())
    for i in range(FRONT_MATTER_OFFSET, _pdf_doc.page_count)
)
book_wide_coverage = words / book_wide_pdf_words if book_wide_pdf_words else 0.0

# Precision/recall on as_p0360, the flagship diagnostic page (plan.md Step 18b): same LCS
# decomposition ocr_f1 uses internally, exposed because the whole diagnosis rests on this
# split -- 81% precision / 28% recall pre-repair meant "accurate but stops early", not
# "misreads", and F1 alone would hide which of the two actually moved.
_p360_pred = (OCR_DIR / "as_p0360.mmd").read_text(encoding="utf-8")
_p360_gold = gold["as_p0360"]
_p_norm, _g_norm = metrics.normalize_latex(_p360_pred), metrics.normalize_latex(_p360_gold)
_overlap = metrics._lcs_length(_p_norm, _g_norm)
p360_precision = _overlap / len(_p_norm) if _p_norm else 0.0
p360_recall = _overlap / len(_g_norm) if _g_norm else 0.0

no_output_rate = len(failures) / N_CONTENT_PAGES

print("=" * 78)
print("STEP 18B -- THE FOUR BEFORE/AFTER NUMBERS (plan.md's own gate)")
print("=" * 78)
print(f"{'metric':<40}{'BEFORE (Step 16)':>18}{'AFTER (this run)':>18}")
print("-" * 78)
print(
    f"{'no-output rate':<40}"
    f"{100 * BEFORE_REPAIR['no_output_rate']:>17.1f}%{100 * no_output_rate:>17.1f}%"
)
print(
    f"{'median page coverage vs PDF text':<40}"
    f"{100 * BEFORE_REPAIR['median_page_coverage']:>17.1f}%{100 * median_coverage:>17.1f}%"
)
print(
    f"{'book-wide word coverage vs PDF text':<40}"
    f"{100 * BEFORE_REPAIR['book_wide_coverage']:>17.1f}%{100 * book_wide_coverage:>17.1f}%"
)
print(
    f"{'as_p0360 precision':<40}"
    f"{BEFORE_REPAIR['as_p0360_precision']:>18.3f}{p360_precision:>18.3f}"
)
print(
    f"{'as_p0360 recall':<40}"
    f"{BEFORE_REPAIR['as_p0360_recall']:>18.3f}{p360_recall:>18.3f}"
)
print("-" * 78)

gate_passed = no_output_rate <= 0.25
print(f"\nGATE (plan.md Step 18b): no-output rate {'<=' if gate_passed else '>'} 25% -> ", end="")
print("PASSED" if gate_passed else "NOT PASSED -- annotation budget reopened, evidence-based")
if not gate_passed:
    print("Decision (plan.md Step 18b DECISION, 2026-08-10): train set expanded 105 -> 122")
    print("pages, targeted at the chapters the repaired reader still failed hardest on --")
    print("ch04_elem_transcend +10, ch24_combinatorial +4, ch29_laplace +3 -- every added")
    print("page is one the repaired reader actually failed on, not an arbitrary pick.")


STEP 18B -- THE FOUR BEFORE/AFTER NUMBERS (plan.md's own gate)
metric                                    BEFORE (Step 16)  AFTER (this run)
------------------------------------------------------------------------------
no-output rate                                       42.9%             28.5%
median page coverage vs PDF text                     28.0%             43.7%
book-wide word coverage vs PDF text                  15.1%             28.3%
as_p0360 precision                                   0.813             0.458
as_p0360 recall                                      0.280             0.833
------------------------------------------------------------------------------

GATE (plan.md Step 18b): no-output rate > 25% -> NOT PASSED -- annotation budget reopened, evidence-based
Decision (plan.md Step 18b DECISION, 2026-08-10): train set expanded 105 -> 122
pages, targeted at the chapters the repaired reader still failed hardest on --
ch04_elem_transcend +10, ch24_combinatorial +4, ch2

## Step 31 — OCR quality (Step 29 TEST set), index statistics, and two retrieval examples

Everything below runs live against this repo's own committed state (`reports/step29_test_eval_results.json`, `data/ocr/`, `data/index/`) -- Restart & Run All reproduces every number, per the grounding gate at the top of this notebook. `data/index/` is gitignored and rebuilt by `bash scripts/build_index.sh` (plan.md Sec.11.6) -- run that first if it isn't present.

In [3]:
# --- Step 31: OCR quality -- Step 29's before/after table, recomputed live from
# reports/step29_test_eval_results.json (the raw per-page results, not the pre-rendered
# .md, so this cell is the grounding-gate evidence, not a copy of it).
import json
from pathlib import Path

STEP29_RESULTS = Path("../reports/step29_test_eval_results.json")
data = json.loads(STEP29_RESULTS.read_text(encoding="utf-8"))
N_TEST_PAGES = 39
for phase in ("baseline", "finetuned", "hybrid"):
    assert len(data[phase]) == N_TEST_PAGES, f"{phase}: expected {N_TEST_PAGES} rows"


def summarize(rows: list[dict]) -> dict:
    n = len(rows)
    failed = [r for r in rows if r["failed"]]
    ok = [r for r in rows if not r["failed"]]
    return {
        "n": n,
        "n_failed": len(failed),
        "failure_rate": len(failed) / n if n else 0.0,
        "char_f1": (sum(r["char_f1"] for r in ok) / len(ok)) if ok else 0.0,
    }


def fail_str(s: dict) -> str:
    return f"{s['n_failed']}/{s['n']} ({s['failure_rate']:.1%})"


base, ft, hyb = (summarize(data[p]) for p in ("baseline", "finetuned", "hybrid"))

print(f"Sample size: {N_TEST_PAGES} held-out TEST pages (grading_kit/labels.jsonl)\n")
print(f"{'metric':<24}{'baseline':>16}{'fine-tuned':>16}{'hybrid':>16}")
print("-" * 72)
print(f"{'failure rate':<24}{fail_str(base):>16}{fail_str(ft):>16}{fail_str(hyb):>16}")
print(f"{'char-F1 (successes)':<24}{base['char_f1']:>16.3f}{ft['char_f1']:>16.3f}{hyb['char_f1']:>16.3f}")
print()
print("Per-page-type breakdown -- hybrid reader (the one Step 30 used for the full-book re-OCR):")
print(f"{'region_type':<16}{'n':>4}{'failure rate':>16}{'char-F1':>10}")
for t in sorted({r["region_type"] for r in data["hybrid"]}):
    rows = [r for r in data["hybrid"] if r["region_type"] == t]
    s = summarize(rows)
    print(f"{t:<16}{s['n']:>4}{fail_str(s):>16}{s['char_f1']:>10.3f}")


Sample size: 39 held-out TEST pages (grading_kit/labels.jsonl)

metric                          baseline      fine-tuned          hybrid
------------------------------------------------------------------------
failure rate               20/39 (51.3%)    9/39 (23.1%)     3/39 (7.7%)
char-F1 (successes)                0.441           0.423           0.443

Per-page-type breakdown -- hybrid reader (the one Step 30 used for the full-book re-OCR):
region_type        n    failure rate   char-F1
formula           18     0/18 (0.0%)     0.500
prose+formula      7      0/7 (0.0%)     0.385
table             14    3/14 (21.4%)     0.387


### Retrieval example 1 — formula/definition queries

Done directly against the FAISS index + chunk sidecar (`data/index/`), not via `retrieval/retriever.py` -- that Stage 5 module is A3 scope and still `raise NotImplementedError` on purpose. A2 only needs to prove the vector index itself is queryable and returns real, citable evidence.

Two queries, not one: the first is the query originally planned for this point -- it did **not** retrieve the expected page, and that finding is reported below rather than discarded. The second is a second, independently-verified query (checked correct before being written here, not adjusted after the fact) that satisfies this point's own "one working retrieval example" requirement -- it lands on the exact expected page and formula, from one of the 39 official TEST pages.

In [4]:
# --- Step 31: one working retrieval example, done live (not via retrieval/retriever.py --
# that Stage 5 module is A3 scope and still `raise NotImplementedError` on purpose; A2 only
# needs to prove the vector index itself is queryable and returns the right evidence).
import json
import sys
from pathlib import Path

sys.path.insert(0, "../src")
import faiss
from sentence_transformers import SentenceTransformer

from doc_agent import config
from doc_agent.contracts import Chunk

cfg = config.load("../configs/config.yaml")

INDEX_DIR = Path("../data/index")
if not (INDEX_DIR / "faiss.index").exists() or not (INDEX_DIR / "chunks.jsonl").exists():
    raise FileNotFoundError(
        f"{INDEX_DIR}/ not built. Run `bash scripts/build_index.sh` first (it reads the "
        "already-committed data/ocr/, so this is a re-embed + re-index pass, not a re-OCR) "
        "-- data/index/ is gitignored on purpose (plan.md Sec.11.6), never shipped."
    )
faiss_index = faiss.read_index(str(INDEX_DIR / "faiss.index"))
index_chunks = [
    Chunk(**json.loads(line), score=0.0)
    for line in (INDEX_DIR / "chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
assert faiss_index.ntotal == len(index_chunks), "index/sidecar mismatch -- rebuild data/index/"
embed_model = SentenceTransformer(cfg["embed"]["model"])


def search(query: str, k: int = 5):
    qvec = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(
        "float32"
    )
    scores, idx = faiss_index.search(qvec, k)
    return [(index_chunks[i], float(s)) for i, s in zip(idx[0], scores[0], strict=True) if i >= 0]


def show(query: str, results):
    print(f'Query: "{query}"')
    for rank, (c, score) in enumerate(results, start=1):
        snippet = c.text.strip().replace("\n", " ")[:160]
        print(f"  #{rank}  score={score:.4f}  chunk_id={c.id}  pages={c.page_ids}")
        print(f"       {snippet}")
    print()


results_1 = search("series expansion of the Bessel function J0")
show("series expansion of the Bessel function J0", results_1)
top_pages_1 = results_1[0][0].page_ids if results_1 else []
right_page = "as_p0360" in top_pages_1
print(
    f"Right page? Expecting as_p0360 (formula 9.1.12) in the top result's pages: "
    f"{'YES' if right_page else 'NO'} -- got {top_pages_1}\n"
)
if not right_page:
    p360 = Path("../data/ocr/as_p0360.mmd").read_text(encoding="utf-8").strip()
    print("Checked why: as_p0360.mmd's actual content only covers formulas 9.1.7-9.1.9")
    print(f"(small-argument limiting forms), not 9.1.12:\n  {p360[:200]}")
    print(
        "\nThe real ascending-series formulas (9.1.10-9.1.13) belong on the next page, "
        "as_p0361 -- whose transcript is present but garbled ('Col_' pattern, mismatched "
        "formula numbers), so it doesn't semantically match the query. Retrieval correctly "
        "found the next-best real match (Bessel-chapter series-expansion content) instead of "
        "failing outright -- but the specific page a human would expect wasn't retrievable, "
        "because the OCR on that page didn't preserve enough signal to match on. This is a "
        "real, unplanned finding, not the confirmation the original expectation assumed."
    )

# --- A second, verified-successful query -- point 2 asks for "one working retrieval
# example... confirmation it is the right page", which the query above did not deliver.
# Rather than adjust the query above until something matched (that would just hide the
# finding above, not fix it), this is a second, independently chosen query, checked to
# land correctly BEFORE being written here -- and it's from as_p0229, one of the 39
# official grading_kit TEST pages (validate.py's ANNOT_TEST_PAGES), not a cherry-picked
# training page.
print("\n" + "=" * 78)
results_3 = search("exponential integral Ei(x)")
show("exponential integral Ei(x)", results_3)
top_chunk_3, top_score_3 = results_3[0]
correct = "as_p0229" in top_chunk_3.page_ids and "5.1.10" in top_chunk_3.id
print(
    f"Right page and formula? Expecting as_p0229 with formula 5.1.10 in the top chunk: "
    f"{'YES' if correct else 'NO'} -- got page(s) {top_chunk_3.page_ids}, chunk {top_chunk_3.id}"
)


Query: "series expansion of the Bessel function J0"
  #1  score=0.6197  chunk_id=ch09_bessel|as_p0363|r00  pages=['as_p0363']
       BESSEL FUNCTION (p.1.72) | |
  #2  score=0.6141  chunk_id=ch26_probability|as_p0932|r00|p01  pages=['as_p0932']
       Asymptotic Expansions (x>0)
  #3  score=0.6059  chunk_id=ch06_gamma|as_p0259|r00|p03  pages=['as_p0259']
       =\frac{\infty}{\beta=1}(n^2+y^2)^{-1} Series Expansions
  #4  score=0.5986  chunk_id=ch09_bessel|as_p0414|r00  pages=['as_p0414']
       BESSEL FUNCTIONS OF INTEGER ORDER (p.7) Table 9.7  BESSEL FUNCTIONS--MISCELLANEOUS ZEROS   s^{\text{th}} Zero of xJ_1(x)-\lambda J_0(x) \lambda/s  1  2  3  4  5
  #5  score=0.5952  chunk_id=ch09_bessel|as_p0415|r00  pages=['as_p0415']
       BESSEL FUNCTIONS OF INTEGER ORDER (p.7) 415 BESSEL FUNCTIONS--MISCELLANEOUS ZEROS  Table 9.7 s^{\text{th}}\,\text{Zero of }J_0(x)Y_0(x)-Y_1(x)J_0(x) \lambda^{-\

Right page? Expecting as_p0360 (formula 9.1.12) in the top result's pages: NO -- got ['as_p0363

### Index statistics (Step 30)

The six numbers form Sec.5 asks for, read live from `data/index/index_meta.json`.

In [5]:
# --- Step 31: index statistics -- Step 30's numbers, read live from data/index/index_meta.json
# (built by `bash scripts/build_index.sh`; not committed, see plan.md Sec.11.6 -- this cell
# assumes it has already been run once, same prerequisite as data/ocr/ for the cell above).
import json
from pathlib import Path

META_PATH = Path("../data/index/index_meta.json")
if not META_PATH.exists():
    raise FileNotFoundError(
        "data/index/ not built yet. Run `bash scripts/build_index.sh` first "
        "(it reads the already-committed data/ocr/, so this is a re-embed + re-index pass, "
        "not a re-OCR)."
    )
meta = json.loads(META_PATH.read_text(encoding="utf-8"))

N_CONTENT_PAGES = 1040
words_indexed = sum(
    len(p.read_text(encoding="utf-8").split()) for p in Path("../data/ocr").glob("*.mmd")
)
ARCHIVE_TOTAL_WORDS = 579_798  # data/provenance.md, sum of page.get_text().split() over all 1082 pages

print("Index statistics (form Sec.5):")
print(f"  n_chunks       : {meta['n_chunks']}")
print(f"  embedding dim  : {meta['embedding_dim']}")
print(f"  index type     : {meta['index_type']}")
print(f"  index size     : {meta['index_size_bytes'] / 1e6:.2f} MB")
print(f"  pages indexed  : {meta['pages_covered']} / {N_CONTENT_PAGES}")
print(f"  chapters       : {meta['chapters_covered']} / 29")
print(f"  embed model    : {meta['embed_model']}")
print(f"  words indexed  : {words_indexed:,} (our OCR) vs {ARCHIVE_TOTAL_WORDS:,} (archive text layer) "
      f"= {100 * words_indexed / ARCHIVE_TOTAL_WORDS:.1f}%")


Index statistics (form Sec.5):
  n_chunks       : 3543
  embedding dim  : 1024
  index type     : faiss:flat
  index size     : 16.22 MB
  pages indexed  : 987 / 1040
  chapters       : 29 / 29
  embed model    : BAAI/bge-m3
  words indexed  : 142,836 (our OCR) vs 579,798 (archive text layer) = 24.6%


### The "words indexed vs total" number, investigated live

The raw ratio above reads like three-quarters of the book is missing. It isn't -- investigated once (Step 30's RESULT in `plan.md`) and reproduced here, not just quoted.

In [6]:
# --- Step 31: "words indexed vs total" investigated, not just quoted (plan.md Step 30
# RESULT + the team's own audit of that number). The naive ratio above (~24.6%) reads like
# three-quarters of the book is missing. It isn't -- two checks, both live:

from pathlib import Path

import pymupdf

# Check 1: chars-per-word ratio on OUR OCR. English prose runs ~5-6 chars/word; a much
# higher ratio is the signature of .split()-based counting undercounting content that has
# no internal whitespace (LaTeX-packed table cells, one giant "word" per cell/column).
mmd_files = sorted(Path("../data/ocr").glob("*.mmd"))
total_chars = sum(len(p.read_text(encoding="utf-8")) for p in mmd_files)
total_words = sum(len(p.read_text(encoding="utf-8").split()) for p in mmd_files)
print(f"Our OCR: {total_chars:,} chars / {total_words:,} words = {total_chars / total_words:.2f} chars/word "
      f"(English prose norm: ~5-6)")

# Check 2: the archive text layer has the SAME artifact, in the OPPOSITE direction -- a
# dense numeric table gets fragmented into extra spurious "words" by the archive's own old
# OCR, inflating the denominator on exactly the pages where ours is undercounted. Concrete
# example: printed page 100 (a table page) in the archive's own text layer --
doc = pymupdf.open("../data/raw/handbookofmathem1964abra.pdf")
FRONT_MATTER_OFFSET = 32
archive_p100 = doc.load_page(100 + FRONT_MATTER_OFFSET - 1).get_text("text").split()
print("\nArchive text layer, printed page 100 (table page), first 10 tokens after the header:")
print(" ", archive_p100[7:17])
print("  -- a single decimal number split across 3 'words' by the archive OCR's own column")
print("  segmentation. This inflates 579,798 the same way LaTeX-packing deflates our count,")
print("  on the very same table-heavy pages -- so the raw ratio is wrong in BOTH directions")
print("  at once, not just one.")

# Check 3: the lowest-word-count SUCCESSFUL pages are exactly the packed-table case, not
# actually near-empty.
per_page_words = sorted(
    ((len(p.read_text(encoding="utf-8").split()), p.stem) for p in mmd_files), key=lambda t: t[0]
)
low = [t for t in per_page_words if t[0] < 50]
print(f"\n{len(low)}/{len(mmd_files)} successful pages ({100 * len(low) / len(mmd_files):.0f}%) "
      f"come in under 50 words by this counting artifact. Lowest: {per_page_words[0]}")
lowest_text = (Path("../data/ocr") / f"{per_page_words[0][1]}.mmd").read_text(encoding="utf-8")
print(f"  {per_page_words[0][1]}.mmd, full content ({len(lowest_text)} chars): {lowest_text.strip()[:200]}")


Our OCR: 1,268,758 chars / 142,836 words = 8.88 chars/word (English prose norm: ~5-6)

Archive text layer, printed page 100 (table page), first 10 tokens after the header:
  ['In', ',r', '0.', '000', '-<»', '0.001', '-6.90775', '52789', '821371', '0.002']
  -- a single decimal number split across 3 'words' by the archive OCR's own column
  segmentation. This inflates 579,798 the same way LaTeX-packing deflates our count,
  on the very same table-heavy pages -- so the raw ratio is wrong in BOTH directions
  at once, not just one.

318/987 successful pages (32%) come in under 50 words by this counting artifact. Lowest: (1, 'as_p1047')
  as_p1047.mmd, full content (364 chars): |p|/|p|=1.00888|p|^{2}|p|^{3}|p|^{4}|p|^{5}|p|^{6}|p|^{7}|p|^{8}|p|^{9}|p|^{10}|p|^{11}|p|^{12}|p|^{13}|p|^{14}|p|^{15}|p|^{16}|p|^{17}|p|^{18}|p|^{19}|p|^{11}|p|^{12}|p|^{13}|p|^{14}|p|^{15}|p|^{16}|


### Retrieval example 2 — table-value lookup (the untested case Step 30 flagged)

Step 30's RESULT block flagged this explicitly: coverage is fine, but nobody had tested whether a table-heavy, LaTeX-packed chunk actually retrieves or reads correctly for an exact-value query. Reporting whichever result comes back.

In [7]:
# --- Step 31 point 4: the SECOND retrieval example -- a table-value lookup, per Step 30's
# flagged-forward open risk. The query above is prose-plus-LaTeX-formula content, which
# Nougat and a text embedder handle normally; this one targets the corpus's actual weak
# point -- table cells that transcribed correctly but got packed into single no-whitespace
# LaTeX tokens (the word-coverage cell above), an atypical input for BAAI/bge-m3. Nobody
# had tested whether that retrieves or reads correctly before this cell. Reporting
# whichever result comes back, success or failure -- that is the point of running it.
results_2 = search("value of J0 at x=1.2")
show("value of J0 at x=1.2", results_2)

top_chunk, top_score = results_2[0]
print(f"Top result: page {top_chunk.page_ids}, chunk {top_chunk.id}, score {top_score:.4f}.")
print("Read it: that is NOT a J0 value or table -- it is unrelated content from a different")
print("chapter (Bernoulli/zeta). Rank #1 is WRONG for this query.\n")

rank2_chunk, rank2_score = results_2[1]
print(f"Rank #2 (score {rank2_score:.4f}, page {rank2_chunk.page_ids}) IS relevant: it is the")
print("polynomial-approximation formula for J0(x) itself -- computable, but a formula to")
print("evaluate, not a pre-computed table value. A real value TABLE for this x range (A&S")
print("Table 9.1, x=0..~2) exists in the corpus (as_p0392/as_p0393) but our own OCR")
print("transcribed it as a near-empty placeholder ('Sample rows: n=10', no actual row data)")
print("-- so the honest answer has two layers: retrieval ranked the right formula 2nd, not")
print("1st, AND the actual ground-truth table for this range was itself under-transcribed")
print("upstream. This is a genuine, reportable failure, not a retrieval implementation bug.")


Query: "value of J0 at x=1.2"
  #1  score=0.6375  chunk_id=ch23_bernoulli_zeta|as_p0805|r02|p01  pages=['as_p0805']
       n=1,2,\ldots,2 \frac{1}{2}>x>0
  #2  score=0.6205  chunk_id=ch09_bessel|as_p0369|r00|p02  pages=['as_p0369']
       J_{0}(x)=1-2.24999 97(x/3)^{2}+1.26562 08(x/3)^{4} -.31638 66(x/3)^{5}+.04444 79(x/3)^{8} -.00394 44 4(x/3)^{10}+.00021 00(x/3)^{12}+\epsilon |e|<5\times 10^{-8
  #3  score=0.6104  chunk_id=ch14_coulomb|as_p0542|r00|14.6.2  pages=['as_p0542']
       14.6.2   L=0,\rho=0
  #4  score=0.6069  chunk_id=ch10_bessel_frac|as_p0452|r04|p00  pages=['as_p0452']
       To identify y_0(x) in terms of xJ_N(x), xY_N(x), restrict x to 0<x\leq b<1 so that by 10.4.118 \xi is negative, and replace the Airy function by its asymptotic 
  #5  score=0.6039  chunk_id=ch22_orthogonal_poly|as_p0802|r02  pages=['as_p0802']
       2\times x | 0.5 | 1.0 (p,x)=0 | \quad x | \

Top result: page ['as_p0805'], chunk ch23_bernoulli_zeta|as_p0805|r02|p01, score 0.6375.
Read it: that is